# CarDekho Dataset: Data Audit & Baseline

## 1. Import libraries and configure paths

In [11]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

DATA_PATH = Path("cardekho_dataset.csv")
OUTPUT_DIR = Path("outputs")
TARGET = "selling_price"
RANDOM_STATE = 42
TEST_SIZE = 0.20

REQUIRED_COLUMNS = [
    "brand", "model", "vehicle_age", "km_driven",
    "seller_type", "fuel_type", "transmission_type",
    "mileage", "engine", "max_power", "seats", TARGET
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load dataset

In [12]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

unnamed = [
    col for col in df.columns
    if str(col).lower().startswith("unnamed:")
]
df = df.drop(columns=unnamed)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (15411, 13)


,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## 3. Dataset structure

In [13]:
structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "unique_values": df.nunique(dropna=True)
})

display(structure)
print("Column names:")
print(df.columns.tolist())

,dtype,non_null_count,unique_values
car_name,object,15411,121
brand,object,15411,32
model,object,15411,120
vehicle_age,int64,15411,24
km_driven,int64,15411,3688
seller_type,object,15411,3
fuel_type,object,15411,5
transmission_type,object,15411,2
mileage,float64,15411,411
engine,int64,15411,110


Column names:
['car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


## 4. Missing values and duplicate rows

In [14]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)

display(missing_report)
print("Exact duplicate rows:", int(df.duplicated().sum()))

missing_report.to_csv(
    OUTPUT_DIR / "missing_values.csv",
    encoding="utf-8-sig"
)

,missing_count,missing_percent,unique_values,dtype
car_name,0,0.0,121,object
brand,0,0.0,32,object
model,0,0.0,120,object
vehicle_age,0,0.0,24,int64
km_driven,0,0.0,3688,int64
seller_type,0,0.0,3,object
fuel_type,0,0.0,5,object
transmission_type,0,0.0,2,object
mileage,0,0.0,411,float64
engine,0,0.0,110,int64


Exact duplicate rows: 167


## 5. Numeric summary

In [15]:
numeric_columns = [
    "vehicle_age", "km_driven", "mileage",
    "engine", "max_power", "seats", TARGET
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_summary = df.select_dtypes(include="number").describe().T
display(numeric_summary)

numeric_summary.to_csv(
    OUTPUT_DIR / "numeric_summary.csv",
    encoding="utf-8-sig"
)

,count,mean,std,min,25%,50%,75%,max
vehicle_age,15411.0,6.036338,3.013291,0.0,4.0,6.00,8.0,29.00
km_driven,15411.0,55616.480631,51618.548422,100.0,30000.0,50000.00,70000.0,3800000.00
mileage,15411.0,19.701151,4.171265,4.0,17.0,19.67,22.7,33.54
engine,15411.0,1486.057751,521.106696,793.0,1197.0,1248.00,1582.0,6592.00
max_power,15411.0,100.588254,42.972979,38.4,74.0,88.50,117.3,626.00
seats,15411.0,5.325482,0.807628,0.0,5.0,5.00,5.0,9.00
selling_price,15411.0,774971.116410,894128.363263,40000.0,385000.0,556000.00,825000.0,39500000.00


## 6. Price outlier screening (IQR)

In [16]:
price = df[TARGET].dropna()

if price.empty:
    raise ValueError("No valid price values were found.")

q1 = price.quantile(0.25)
q3 = price.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = ((price < lower) | (price > upper)).sum()

print(f"Q1: {q1:,.2f}")
print(f"Q3: {q3:,.2f}")
print(f"Lower bound: {lower:,.2f}")
print(f"Upper bound: {upper:,.2f}")
print(f"Potential price outliers: {outliers:,}")

Q1: 385,000.00
Q3: 825,000.00
Lower bound: -275,000.00
Upper bound: 1,485,000.00
Potential price outliers: 1,386


## 7. Prepare modeling data

In [17]:
missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

model_df = df[
    ["brand", "model", "vehicle_age", TARGET]
].copy()

initial_rows = len(model_df)

model_df = model_df.dropna(
    subset=["brand", "model", "vehicle_age", TARGET]
)

model_df = model_df[
    (model_df[TARGET] > 0)
    & (model_df["vehicle_age"] >= 0)
    & (model_df["brand"].astype(str).str.strip() != "")
    & (model_df["model"].astype(str).str.strip() != "")
].copy()

print("Rows before filtering:", initial_rows)
print("Rows after filtering:", len(model_df))
print("Rows removed:", initial_rows - len(model_df))

if len(model_df) < 5:
    raise ValueError("Too few valid rows for evaluation.")

display(model_df.head())

Rows before filtering: 15411
Rows after filtering: 15411
Rows removed: 0


,brand,model,vehicle_age,selling_price
0,Maruti,Alto,9,120000
1,Hyundai,Grand,5,550000
2,Hyundai,i20,11,215000
3,Maruti,Alto,9,226000
4,Ford,Ecosport,6,570000


## 8. Train/test split

In [18]:
train, test = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Training rows:", len(train))
print("Test rows:", len(test))

Training rows: 12328
Test rows: 3083


## 9. Fit Baseline


In [19]:
median_brand_model_age = train.groupby(
    ["brand", "model", "vehicle_age"]
)[TARGET].median()

median_brand_model = train.groupby(
    ["brand", "model"]
)[TARGET].median()

median_brand = train.groupby("brand")[TARGET].median()
global_median = train[TARGET].median()

def predict_baseline(row):
    key3 = (
        row["brand"],
        row["model"],
        row["vehicle_age"]
    )
    key2 = (row["brand"], row["model"])
    key1 = row["brand"]

    if key3 in median_brand_model_age.index:
        return median_brand_model_age.loc[key3]

    if key2 in median_brand_model.index:
        return median_brand_model.loc[key2]

    if key1 in median_brand.index:
        return median_brand.loc[key1]

    return global_median

y_true = test[TARGET]
y_pred = test.apply(predict_baseline, axis=1)

## 10. Evaluate Baseline using MAE

In [20]:
mae = mean_absolute_error(y_true, y_pred)

print(f"Baseline MAE: {mae:,.2f}")
print(f"Training global median: {global_median:,.2f}")

results = test[
    ["brand", "model", "vehicle_age", TARGET]
].copy()

results["predicted_price"] = y_pred
results["absolute_error"] = (
    results[TARGET] - results["predicted_price"]
).abs()

display(results.head(10))

results.to_csv(
    OUTPUT_DIR / "baseline_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics = pd.DataFrame({
    "metric": [
        "dataset_rows",
        "modeling_rows",
        "training_rows",
        "test_rows",
        "random_state",
        "test_size",
        "training_global_median",
        "baseline_MAE"
    ],
    "value": [
        len(df),
        len(model_df),
        len(train),
        len(test),
        RANDOM_STATE,
        TEST_SIZE,
        global_median,
        mae
    ]
})

display(metrics)

metrics.to_csv(
    OUTPUT_DIR / "baseline_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

Baseline MAE: 113,896.89
Training global median: 560,000.00


,brand,model,vehicle_age,selling_price,predicted_price,absolute_error
3334,Hyundai,i10,12,190000,210000.0,20000.0
10928,Maruti,Baleno,4,600000,628500.0,28500.0
2518,Maruti,Ertiga,7,665000,611000.0,54000.0
11322,Honda,City,1,1570000,1170000.0,400000.0
9394,Maruti,Alto,11,160000,170000.0,10000.0
13649,Tata,Safari,10,675000,285000.0,390000.0
11892,Hyundai,Grand,7,465000,400000.0,65000.0
5133,Datsun,RediGO,4,260000,260000.0,0.0
1611,Honda,City,11,300000,328000.0,28000.0
8137,BMW,X1,9,850000,1195000.0,345000.0


,metric,value
0,dataset_rows,15411.000000
1,modeling_rows,15411.000000
2,training_rows,12328.000000
3,test_rows,3083.000000
4,random_state,42.000000
5,test_size,0.200000
6,training_global_median,560000.000000
7,baseline_MAE,113896.894259
